# Robot arm, live testing 24.02.2026

### Connect to the simulator / to the robot

------ simulation only ----------

In [1]:
import sys
from pathlib import Path
from math import cos, sin, pi, radians

# Make sure ur3e-control is on the path
sys.path.insert(0, str(Path.cwd().parents[1]))

from my_simulation.iscoin_sim.kinematics import (
    analytical_ik,
    forward_kinematics_matrix,
    pose_to_matrix,
    matrix_to_tcp6d,
)

from my_simulation.iscoin_sim import ISCoinSim as ISCoin
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor


# # Connect to the simulator
iscoin = ISCoin()
robot = iscoin.robot_control
print("Connected to simulator")

ISCoinSim connected to container 'iscoin_simulator'
Connected to simulator


----------- REAL ROBOT ONLY ----------------

In [2]:
from URBasic import ISCoin
import sys
from pathlib import Path
from math import cos, sin, pi, radians
from my_simulation.iscoin_sim import ISCoinSim as ISCoin
from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor

from my_simulation.iscoin_sim.kinematics import (
    analytical_ik,
    forward_kinematics_matrix,
    pose_to_matrix,
    matrix_to_tcp6d,
)



# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
iscoin = ISCoin(host="10.30.5.158", opened_gripper_size_mm=40)

In [3]:
robot = iscoin.robot_control

## Default home position

This joint position is equivalent to this one:
```
home_tcp = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.20,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )
```



In [2]:
# Go to home position
home = Joint6D.createFromRadians(1.1859, -1.4452, 1.2389, -1.3639, -1.5693, -0.3849)
robot.movej(home, a=radians(80), v=radians(60))

tcp_home = robot.get_actual_tcp_pose()
print(f"Home TCP: {tcp_home}")

movej sent (duration=3s)
Movement OK — target reached
Home TCP: TCP6D([-0.00024157263562829242, -0.3499990478240251, 0.345002652980121, -5.0664473506716306e-06, 3.139981903409545, -1.4319116867048077e-05])


In [ ]:
# target_tcp = TCP6D.createFromMetersRadians(
#       0.0,      # x
#      -0.35,     # y
#       0.20,     # z
#       0.0,      # rx
#       3.14,      # ry
#       0.0       # rz
#   )

In [ ]:
# current_joints = robot.get_actual_joint_positions()
# target_joints = robot.get_inverse_kin(target_tcp, qnear=current_joints)

## Set TCP

In [3]:
pen_length = 0.2385  # meters
robot.set_tcp(TCP6D.createFromMetersRadians(0, 0, pen_length, 0, 0, 0))

In [4]:
tcp = robot.get_actual_tcp_pose()
print(tcp)

TCP6D([0.00014259111527307073, -0.3500012224442032, 0.10650296238601667, -5.0664473506716306e-06, 3.139981903409545, -1.4319116867048077e-05])


## Verify the Z plan for 2d drawing

In [ ]:
robot.reset_error()

In [ ]:
# Enable — you can physically move the arm by hand
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [ ]:
p1 = robot.get_actual_tcp_pose()
print(p1)

In [ ]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [ ]:
p2 = robot.get_actual_tcp_pose()
print(p2)

In [ ]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [ ]:
p3 = robot.get_actual_tcp_pose()
print(p3)

Verification of z values

In [ ]:
print(f"Z values: {p1.z:.4f}, {p2.z:.4f}, {p3.z:.4f}")
print(f"Z spread: {max(p1.z, p2.z, p3.z) - min(p1.z, p2.z, p3.z):.4f} m")

# Gripper setup ( manually )

In [23]:
iscoin.gripper.activate()

True

In [76]:
iscoin.gripper.deactivate()

True

In [ ]:
from time import sleep
if not iscoin.gripper.open():
  print("Failed to open gripper")
else:
  sleep(1)
  if not iscoin.gripper.close():
    print("Failed to close gripper")


In [75]:
iscoin.gripper.open()

True

In [77]:
iscoin.gripper.close()

False

### Drawing algorithm live

Drawings parameters

In [7]:
import numpy as np

drawing_z = 0.0014
safe_z = 0.13

rx, ry, rz = 0.0, 3.14, 0.0

surface_tilt_y = radians(45)  # radians — e.g. radians(45) to tilt 45° around Y

draw_speed = 0.05   # m/s — slow for drawing
travel_speed = 0.1   # m/s — faster for travel moves

After testing: drawing_z = 0.0015 is good

In [8]:
sim_ref_point = TCP6D.createFromMetersRadians(
      0.0,      # x
     -0.35,     # y
      0.13,     # z
      0.0,      # rx
      3.14,      # ry
      0.0       # rz
  )

In [9]:
robot.movel(sim_ref_point, v=travel_speed)

movej sent (duration=2s)
Movement OK — target reached


True

Set drawing reference point

In [14]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

# Take x, y from where you placed the arm, z and orientation from drawing parameters
ref = robot.get_actual_tcp_pose()
ref_drawing_point = TCP6D.createFromMetersRadians(ref.x, ref.y, drawing_z, rx, ry, rz)
print(f"Drawing ref: x={ref.x:.4f}, y={ref.y:.4f}, z={drawing_z}")

In [10]:
#### FOR SIMULATION

ref_drawing_point = TCP6D.createFromMetersRadians(0, -0.35, safe_z, rx, ry, rz)

Build the surface transform (run after either real or simulation ref point)

In [11]:
# Rotation around Y
c, s = cos(surface_tilt_y), sin(surface_tilt_y)
R_tilt = np.eye(4)
R_tilt[0, 0] = c;  R_tilt[0, 2] = s
R_tilt[2, 0] = -s; R_tilt[2, 2] = c

print("R_tilt =")
print(R_tilt)

R_tilt =
[[ 0.70710678  0.          0.70710678  0.        ]
 [ 0.          1.          0.          0.        ]
 [-0.70710678  0.          0.70710678  0.        ]
 [ 0.          0.          0.          1.        ]]


In [12]:
# Translation
T_trans = np.eye(4)
T_trans[0, 3] = ref_drawing_point.x
T_trans[1, 3] = ref_drawing_point.y
T_trans[2, 3] = ref_drawing_point.z

print("T_trans =")
print(T_trans)

T_trans =
[[ 1.    0.    0.    0.  ]
 [ 0.    1.    0.   -0.35]
 [ 0.    0.    1.    0.13]
 [ 0.    0.    0.    1.  ]]


In [13]:
# Compose: rotate local coords first, then translate to world
T_surface = T_trans @ R_tilt
print(f"Surface origin: x={ref_drawing_point.x:.4f}, y={ref_drawing_point.y:.4f}, tilt_y={surface_tilt_y:.2f} rad")
print()
print("T_surface =")
print(T_surface)

Surface origin: x=0.0000, y=-0.3500, tilt_y=0.79 rad

T_surface =
[[ 0.70710678  0.          0.70710678  0.        ]
 [ 0.          1.          0.         -0.35      ]
 [-0.70710678  0.          0.70710678  0.13      ]
 [ 0.          0.          0.          1.        ]]


Generate trajectory

In [ ]:
robot.freedrive_mode()

In [ ]:
robot.end_freedrive_mode()

In [14]:
# Circle: radius 5cm, 32 segments
r = 0.05
n = 32

path_offsets = [(r * cos(2 * pi * i / n), r * sin(2 * pi * i / n)) for i in range(n + 1)]
print(f"{len(path_offsets)} offsets generated")

33 offsets generated


In [12]:
# Lemniscate of Bernoulli: width 10cm, 64 segments
a = 0.05
n = 64

path_offsets = []
for i in range(n + 1):
    t = 2 * pi * i / n
    denom = 1 + sin(t) ** 2
    dx = a * cos(t) / denom
    dy = a * sin(t) * cos(t) / denom
    path_offsets.append((dx, dy))

print(f"{len(path_offsets)} offsets generated (lemniscate)")

65 offsets generated (lemniscate)


## Convert local coordinates to world TCP

In [15]:
def local_to_world(T_surface, lx, ly, lz=0.0):
    """Transform a local 2D point to a world TCP using surface transform."""
    world = T_surface @ np.array([lx, ly, lz, 1.0])
    return TCP6D.createFromMetersRadians(world[0], world[1], world[2], rx, ry + surface_tilt_y, rz)

waypoints = []
for i, (lx, ly) in enumerate(path_offsets):
    tcp = local_to_world(T_surface, lx, ly, lz=drawing_z)
    blend = 0.005 if i < len(path_offsets) - 1 else 0.0
    waypoints.append(TCP6DDescriptor(tcp, v=draw_speed, r=blend))

print(f"{len(waypoints)} waypoints ready")

33 waypoints ready


Move to start above

In [16]:
start_above = local_to_world(T_surface, path_offsets[0][0], path_offsets[0][1], lz=safe_z - 0.10)
robot.movel(start_above, v=travel_speed)

movej sent (duration=2s)
Movement OK — target reached


True

In [17]:
test_point = robot.get_actual_tcp_pose()
print(test_point)

TCP6D([0.056568542494394886, -0.3499999999979458, 0.11585786437397733, 4.363009573652465e-12, -2.3577871437791504, -1.2080201953923997e-11])


Move down to paper (A)

In [22]:
start_down = local_to_world(T_surface, path_offsets[0][0], path_offsets[0][1], lz=drawing_z)
robot.movel(start_down, v=travel_speed)

movej sent (duration=2s)
Movement OK — target reached


True

In [36]:
test_point = robot.get_actual_tcp_pose()
print(test_point)

TCP6D([0.050000000000460024, -0.3499999999981451, 0.0013999999987282241, -2.1370395896197135e-14, 3.139999999835178, 8.828682836881377e-12])


Draw

In [23]:
robot.movel_waypoints(waypoints)

movel_waypoints sent (33 points, total=66s)
Movement OK — target reached


True

 Lift up and go home

In [20]:
end_above = local_to_world(T_surface, path_offsets[-1][0], path_offsets[-1][1], lz=safe_z)
robot.movel(end_above, v=travel_speed)

INFO: Analytical IK returned 0 valid solutions for target at distance=0.4165m (max reach ~0.457m)
ERROR: movel failed — could not find inverse kinematics solution


False

In [21]:
robot.movej(home, a=radians(80), v=radians(60))

movej sent (duration=3s)
Movement OK — target reached


True